# Projeto 01 — Agente Inteligente de Otimização de Rotas de Distribuição Alimentar Local
## Unidade Curricular de Inteligência Artificial — IPCA (2026/2027)
### Resolução de Problemas de Pickup-and-Delivery com Janelas Temporais e Capacidade (PDPTW)

---

### Grupo 04 — `IA26_P01_G04`
* **Gonçalo Correia** — Nº 22840
* **Bruno Dantas** — Nº 27958
* **Ricardo Ferreira** — Nº 25426

**Repositório Oficial:** [https://github.com/BrunoDantas1/IA26_P01_G04.git](https://github.com/BrunoDantas1/IA26_P01_G04.git)

---

## 1. Introduction

### 1.1. Contexto do Problema e Enquadramento
A transição para cadeias de abastecimento alimentar curtas e sustentáveis coloca desafios logísticos complexos. Pequenos produtores agrícolas locais operam com margens temporais reduzidas de frescura dos produtos, enquanto os consumidores finais e mercados municipais exigem entregas em horários de conveniência bem definidos.

Neste projeto desenvolvemos um **Agente Inteligente de Planeamento de Rotas** para distribuição alimentar local na região do distrito de **Braga** e concelho da **Póvoa de Varzim**. O problema enquadra-se formalmente na classe dos **Pickup-and-Delivery Problems with Time Windows and Capacity Constraints (PDPTW)**, que combinam restrições espaciais, de precedência, temporais e de capacidade.

### 1.2. Atores e Componentes do Sistema
1. **Produtores Locais (*Producers*)**: Locais de origem onde os produtos agrícolas são colhidos/embalados e disponibilizados dentro da janela regulamentar $[\text{availability\_start}, \text{availability\_end}]$.
2. **Centros de Recolha / Armazéns (*CollectionCenters*)**: Plataformas logísticas centrais onde os bens podem ser agregados e que servem de depósito base permanente da frota de veículos.
3. **Locais de Entrega (*DeliveryLocations / Customers*)**: Clientes residenciais ou mercados municipais com janelas de entrega $[\text{delivery\_start}, \text{delivery\_end}]$.
4. **Frota de Veículos (*Vehicles*)**: Carrinhas comerciais com capacidades máximas específicas (1444.7 kg e 1009.8 kg) que realizam as viagens.
5. **Matriz de Distâncias e Tempos (*DistanceMatrix*)**: Baseada na rede de paragens unificadas de `locations.csv`, fornecendo distâncias em km (distância ortodrómica corrigida com fator rodoviário de 1.3) e tempos de viagem em minutos calculados a uma velocidade de 45 km/h.

### 1.3. Objetivos do Projeto e Divisão de Tarefas
* **Secção 1 — Introdução**: Contextualização, identificação e definição do sistema logístico (preenchida).
* **Secção 2 — Exploratory Data Analysis (EDA)**: Carregamento do dataset, validação de integridade referencial e análise da procura face à frota *(incluída nesta entrega)*.
* **Secção 3 — Modelação e Testes de Viabilidade**: Definição das classes `Request`, `Vehicle`, `State`, `Problem`, função de custo e validador estrito de rotas com testes manuais (*hand-written routes*) *(incluída nesta entrega)*.
* **Secção 4 — Problem Solving**: Parte A — pesquisa informada com Greedy e A*, incluindo duas heurísticas e comparação experimental. Pesquisa local (Hill-Climbing) e CSP com coordenação de frota ficam reservados às restantes partes do grupo.
* **Secção 5 — Result Analysis & Conclusion**: Análise global e conclusões a completar após integração das restantes partes.


## 2. Exploratory Data Analysis (EDA)

Os valores seguintes foram confirmados nos 11 CSV de `data`. A análise completa consta de `EDA.ipynb` e `01_eda_tiny.ipynb`.

### 2.1 Dimensões e integridade
| Tabela | Registos |
|---|---:|
| localities | 49 |
| producers | 5 |
| products | 23 |
| collection_centers | 1 |
| vehicles | 2 |
| customers | 8 |
| delivery_locations | 6 |
| orders | 10 |
| order_items | 23 |
| locations | 12 |
| distance_matrix | 132 |

As 10 PK simples e a chave composta da matriz são únicas e não nulas. As 12 relações FK verificadas e os mapeamentos de `locations.ref_id` são válidos. Há 132 = 12 × 11 pares dirigidos, excluindo a diagonal; o modelo atribui distância zero à diagonal.

### 2.2 Procura e capacidade
São **23 artigos de 10 encomendas**, com **248,96 kg** no total. Veículo 1: **1444,70 kg**; veículo 2: **1009,80 kg**; centro: **3232,90 kg**. Ambos os veículos têm `refrigerated=0`.

A procura representa **17,23%** da capacidade do veículo 1 ou **24,65%** do veículo 2. Capacidade em massa suficiente não demonstra uma rota única viável. O máximo por data de abertura da recolha é **43,58 kg em 19/10/2026**.

### 2.3 Janelas temporais
`ready_time = max(order_datetime, availability_start)`. Encomendar após a abertura é permitido, desde que a prontidão não ultrapasse o fecho. **12 artigos** têm prontidão após o fecho: **1, 3, 4, 9, 13, 14, 15, 16, 18, 20, 21 e 23**.

Os outros **11 artigos** são candidatos à recolha, sem garantia de viabilidade da rota. As janelas de entrega duram **1 a 4 horas** e podem ocorrer no dia seguinte à disponibilidade. Os artigos 7, 10 e 22 partilham a data de abertura, mas isso não garante uma rota conjunta viável.

### 2.4 Geografia e matriz
O centro fica em **Terras de Bouro**. Existem cinco produtores e seis destinos; as distâncias do centro aos destinos variam entre **9,14 km e 50,94 km**. O modelo usa as distâncias e durações fornecidas no CSV; as durações são compatíveis, por arredondamento, com cerca de **45 km/h**.

### 2.5 Consequências para o planeamento
Preservar os artigos originais e sinalizar os inviáveis. Validar cada rota com deslocações, espera, capacidade, precedência e regresso ao depósito. Integridade relacional e capacidade agregada não demonstram viabilidade temporal.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

tables = {p.stem: pd.read_csv(p) for p in Path('data').glob('*.csv')}
expected = dict(localities=49, producers=5, products=23, collection_centers=1,
                vehicles=2, customers=8, delivery_locations=6, orders=10,
                order_items=23, locations=12, distance_matrix=132)
assert {name: len(frame) for name, frame in tables.items()} == expected
items = tables['order_items'].merge(tables['orders'], on='order_id').merge(tables['products'], on='product_id')
weight = (items.quantity * items.unit_weight_kg).sum()
ready = pd.concat([pd.to_datetime(items.order_datetime), pd.to_datetime(items.availability_start)], axis=1).max(axis=1)
anomalies = sorted(items.loc[ready > pd.to_datetime(items.availability_end), 'order_item_id'])
assert np.isclose(weight, 248.96)
assert anomalies == [1, 3, 4, 9, 13, 14, 15, 16, 18, 20, 21, 23]
assert tables['vehicles'].refrigerated.eq(0).all()
print(f'CSV confirmados: {weight:.2f} kg, {len(anomalies)} artigos com prontidão após o fecho.')


CSV confirmados: 248.96 kg, 12 artigos com prontidão após o fecho.


## 3. Modeling & Feasibility Tests

`model.py` define `Request`, `Vehicle`, `RouteStep`, `State` e `Problem`. O estado acompanha localização, hora, carga, artigos recolhidos/entregues e custos acumulados.

`actions` respeita capacidade, precedência e início efetivo do serviço após espera. `return_to_base` exige carga vazia e ligação viável. `is_goal` exige os artigos alvo entregues e regresso explícito ao depósito do veículo.

`route_feasibility_checker` exige partida inicial e regresso final ao depósito. Rejeita ações de depósito intermédias, duplicados, entregas sem recolha, carga por entregar, excesso de capacidade, janelas ultrapassadas após espera e ligações inválidas. Os limites temporais são inclusivos; o serviço tem duração zero.

`cost_function` aplica a mesma validação e calcula distância, viagem, espera e duração total. Uma rota inválida devolve `feasible=False` e custo infinito.

As duas rotas manuais válidas mantêm **164,51 km / 241,70 min** e **263,23 km / 351,00 min**. As demonstrações abaixo apresentam dois casos válidos e três inválidos: precedência, janela temporal e capacidade.


In [2]:
import runpy

runpy.run_path('test_handwritten_routes.py', run_name='__main__');


Pedidos válidos: 11

--- Teste Rota Válida 1 (Item 5) ---
Viável: True | Rota perfeitamente viável! Cumpre todas as restrições.
Distância: 164.51 km | Tempo: 241.7 min (Viagem: 219.4 min, Espera: 22.3 min)
  pickup (Item 5) @ Loc 4: Chegada 2026-10-10 13:26:42 -> Saída 2026-10-10 13:26:42 | Espera: 0.0m | Carga: 1.4kg
  delivery (Item 5) @ Loc 10: Chegada 2026-10-10 14:37:42 -> Saída 2026-10-10 15:00:00 | Espera: 22.3m | Carga: 0.0kg
  return (Item 6) @ Loc 6: Chegada 2026-10-10 16:01:42 -> Saída 2026-10-10 16:01:42 | Espera: 0.0m | Carga: 0.0kg

--- Teste Rota Válida 2 (Itens 10 e 7) ---
Viável: True | Rota perfeitamente viável! Cumpre todas as restrições.
Distância: 263.23 km | Tempo: 351.0 min

--- Teste Rota Inválida 1 (Precedência) ---
Viável: False | Violação de Precedência no passo 1: Tentativa de entrega do Item 5 sem ter sido previamente recolhido!

--- Teste Rota Inválida 2 (Janela Temporal) ---
Viável: False | Violação de Janela Temporal de Recolha no passo 1 (Item 5): Iníci

## 4. Problem Solving (Algoritmos de IA)

### 4.1 Parte A — Pesquisa informada

O objetivo desta parte é encontrar uma rota viável de menor distância, com partida e regresso ao depósito. Comparamos Greedy Best-First Search e A* com duas heurísticas, reutilizando as funções de `informed.py` e o modelo existente. Os algoritmos não são duplicados neste notebook.

#### 4.1.1 Cenário e configuração

Selecionámos os artigos **7 e 10**, com partida em **19/10/2026 às 07:30**, do depósito **6**. Ambos têm recolha no local 4 e entrega no próprio dia. O cenário permite observar decisões de ordenação, precedência e janelas temporais sem tornar a pesquisa inicial demasiado extensa.

| Artigo | Peso | Recolha | Entrega |
|---|---:|---|---|
| 7 | 21,10 kg | Local 4, 08:30–13:30 | Local 10, 11:30–12:30 |
| 10 | 6,54 kg | Local 4, 07:30–09:30 | Local 8, 08:30–12:30 |

A procura selecionada é **27,64 kg**. O veículo 1 tem capacidade de **1444,70 kg** e o veículo 2 de **1009,80 kg**. Ambos pertencem ao mesmo centro, partem do local 6 e não têm refrigeração. As experiências são independentes: cada veículo transporta os mesmos dois artigos numa execução própria; não existe repartição de pedidos nem coordenação simultânea da frota nesta parte.

O dataset contém 12 artigos com prontidão posterior ao fecho da recolha. Os artigos selecionados não apresentam esse problema e têm uma solução viável conhecida. A seleção é explícita e feita apenas em memória, sem modificar os CSV. Filtrar só pela data não seria suficiente: há outros artigos com recolha nesta data e entrega no dia seguinte, ou com prontidão incompatível com a recolha.

O estado inicial tem localização 6, carga zero, nenhum artigo recolhido ou entregue e custos acumulados nulos. O objetivo exige ambos os artigos entregues, carga vazia e regresso explícito ao depósito. `actions()` gera os sucessores e incorpora deslocações, espera e restrições. As durações vêm da matriz CSV; as velocidades individuais de `Vehicle` não são usadas para recalculá-las.

In [3]:
from time import perf_counter
from IPython.display import display
from model import Problem
from informed import (
    greedy_best_first, a_star_search,
    heuristic, heuristic_mst, reconstruct_route,
)

problem_a = Problem(data_folder="data")
target_items_a = {7, 10}
start_time_a = pd.Timestamp("2026-10-19 07:30:00")
problem_a.requests = [r for r in problem_a.requests if r.item_id in target_items_a]
problem_a.requests_by_id = {r.item_id: r for r in problem_a.requests}

vehicle_configuration_a = pd.DataFrame([
    {"Veículo": v.vehicle_id, "Matrícula": v.plate, "Depósito": v.center_location_id,
     "Capacidade (kg)": v.capacity_kg, "Refrigerado": v.refrigerated,
     "Velocidade declarada (km/h)": v.avg_speed_kmh}
    for v in problem_a.vehicles.values()
])
display(vehicle_configuration_a)
display(pd.DataFrame([
    {"Artigo": r.item_id, "Peso (kg)": r.weight_kg,
     "Recolha": r.pickup_location_id, "Prontidão": r.ready_time,
     "Fim de recolha": r.availability_end, "Entrega": r.delivery_location_id,
     "Início de entrega": r.delivery_start, "Fim de entrega": r.delivery_end}
    for r in problem_a.requests
]))

,Veículo,Matrícula,Depósito,Capacidade (kg),Refrigerado,Velocidade declarada (km/h)
0,1,ZG-06-DP,6,1444.7,False,57.8
1,2,TY-74-YA,6,1009.8,False,59.5


,Artigo,Peso (kg),Recolha,Prontidão,Fim de recolha,Entrega,Início de entrega,Fim de entrega
0,7,21.10,4,2026-10-19 08:30:00,2026-10-19 13:30:00,10,2026-10-19 11:30:00,2026-10-19 12:30:00
1,10,6.54,4,2026-10-19 07:30:00,2026-10-19 09:30:00,8,2026-10-19 08:30:00,2026-10-19 12:30:00


#### 4.1.2 Algoritmos e heurísticas

**Greedy Best-First Search** ordena a fronteira apenas por $h(n)$, a estimativa da distância restante. Não considera a distância já percorrida na prioridade e não garante a solução ótima.

**A estrela** ordena a fronteira por $f(n)=g(n)+h(n)$, onde $g(n)$ é `state.accumulated_distance_km`. Neste espaço finito, com custos não negativos e heurística admissível, a primeira solução retirada da fronteira tem distância mínima. Não se garante a duração mínima, pois o objetivo de otimização é a distância.

Ambos usam `heapq`, com um contador para desempatar prioridades iguais. A pesquisa mantém percursos distintos; não elimina estados só por coincidirem na localização, pois a hora e os artigos transportados também condicionam a viabilidade. Cada artigo só pode ser recolhido e entregue uma vez, e o regresso termina a rota: não há ciclos de ações neste cenário. Esta opção é simples para dois artigos, mas pode repetir estados equivalentes e crescer rapidamente com mais pedidos.

Um nó é contado como expandido quando a pesquisa chama `actions()` para gerar os seus sucessores, mesmo que não existam sucessores. O estado objetivo não é expandido.

**Heurística 1 — `heuristic()`**. Para cada artigo ainda não entregue, estima a distância da posição atual até à recolha, se faltar, depois até à entrega e ao depósito. Para artigos já recolhidos, omite a recolha. Usa o máximo destas cadeias individuais e da distância direta ao depósito, em vez de as somar, para evitar contar deslocações partilhadas várias vezes.

**Heurística 2 — `heuristic_mst()`**. Reúne a posição atual, o depósito e os locais de recolha e entrega ainda obrigatórios, sem duplicados. Calcula por Prim a árvore de expansão mínima que os liga. Cada ligação usa o menor custo entre os dois sentidos. Ignora a ordem das visitas e as restrições temporais e de capacidade.

**Admissibilidade.** A primeira heurística é um limite inferior porque qualquer solução tem de cumprir cada cadeia individual obrigatória. A segunda também o é: uma rota que visita todos os locais custa pelo menos a árvore mínima que os liga. Estes argumentos usam as distâncias não negativas e a desigualdade triangular do dataset atual; a célula seguinte verifica essa propriedade. Retirar restrições de precedência ou tempo na estimativa não aumenta o custo mínimo. Ambas devolvem zero no objetivo. Não se deve generalizar a garantia a uma matriz diferente sem voltar a verificar estas condições.

A primeira heurística considera cadeias de artigos; a segunda considera a ligação global dos locais. A árvore mínima exige mais cálculos por avaliação. Neste cenário, a primeira dá inicialmente **164,51 km** e a segunda **102,84 km**; uma estimativa maior, desde que admissível, fornece um limite inferior mais apertado nesse estado. Isso não demonstra dominância em todos os estados.

In [4]:
locations_a = list(problem_a.locations_df.location_id)
distances_a = problem_a.distance_km
assert all(np.isfinite(d) and d >= 0 for d in distances_a.values())
assert all(
    distances_a[i, k] <= distances_a[i, j] + distances_a[j, k] + 1e-9
    for i in locations_a for j in locations_a for k in locations_a
)
print(f"Desigualdade triangular confirmada nas {len(locations_a) ** 3} combinações de localizações.")
for vehicle_id_a in (1, 2):
    initial_a = problem_a.initial_state(vehicle_id_a, start_time_a)
    print(f"Veículo {vehicle_id_a}: h1={heuristic(problem_a, initial_a, vehicle_id_a):.2f} km; "
          f"h2={heuristic_mst(problem_a, initial_a, vehicle_id_a):.2f} km")

Desigualdade triangular confirmada nas 1728 combinações de localizações.
Veículo 1: h1=164.51 km; h2=102.84 km
Veículo 2: h1=164.51 km; h2=102.84 km


#### 4.1.3 Execução e validação das oito combinações

Executamos os dois algoritmos com cada heurística, para cada veículo, mantendo os mesmos pedidos e hora de partida. Cada função recebe a heurística como parâmetro. O tempo em milissegundos mede apenas a chamada de pesquisa; exclui carregamento dos dados, validação, cálculo final de métricas e apresentação.

A rota guardada no estado é convertida por `reconstruct_route()`, que acrescenta a partida explícita e produz pares ação/alvo. Cada solução é confirmada por `route_feasibility_checker()` e as métricas são obtidas por `cost_function()`. As rotas são apresentadas separadamente e a tabela permite comparar distância, duração, nós expandidos e tempo medido. É feita uma medição por combinação, suficiente para demonstrar a execução, mas insuficiente para concluir qual método é mais rápido.

In [5]:
results_a = []
for algorithm_name_a, search_a in (("Greedy", greedy_best_first), ("A*", a_star_search)):
    for estimate_a in (heuristic, heuristic_mst):
        for vehicle_id_a in (1, 2):
            started_a = perf_counter()
            goal_a, expanded_a = search_a(problem_a, vehicle_id_a, start_time_a, estimate_a)
            elapsed_ms_a = (perf_counter() - started_a) * 1000
            if goal_a is None:
                raise RuntimeError(f"Sem solução: {algorithm_name_a}, {estimate_a.__name__}, veículo {vehicle_id_a}")
            depot_a = problem_a.vehicles[vehicle_id_a].center_location_id
            route_a = reconstruct_route(goal_a, depot_a)
            feasible_a, reason_a, _ = problem_a.route_feasibility_checker(
                route_a, vehicle_id_a, start_time_a
            )
            if not feasible_a:
                raise RuntimeError(reason_a)
            metrics_a = problem_a.cost_function(route_a, vehicle_id_a, start_time_a)
            print(f"{algorithm_name_a} | {estimate_a.__name__} | Veículo {vehicle_id_a}")
            print(f"Rota: {route_a}")
            print(f"Validação: {reason_a}\n")
            results_a.append({
                "Algoritmo": algorithm_name_a, "Heurística": estimate_a.__name__,
                "Veículo": vehicle_id_a, "Distância (km)": metrics_a["total_distance_km"],
                "Duração (min)": metrics_a["total_time_min"],
                "Nós expandidos": expanded_a, "Pesquisa (ms)": elapsed_ms_a,
                "Viável": feasible_a,
            })
comparison_a = pd.DataFrame(results_a)
display(comparison_a.round({"Distância (km)": 2, "Duração (min)": 2, "Pesquisa (ms)": 3}))

Greedy | heuristic | Veículo 1
Rota: [('start', 6), ('recolha', 7), ('recolha', 10), ('entrega', 7), ('entrega', 10), ('regresso', 6)]
Validação: Rota perfeitamente viável! Cumpre todas as restrições.

Greedy | heuristic | Veículo 2
Rota: [('start', 6), ('recolha', 7), ('recolha', 10), ('entrega', 7), ('entrega', 10), ('regresso', 6)]
Validação: Rota perfeitamente viável! Cumpre todas as restrições.

Greedy | heuristic_mst | Veículo 1
Rota: [('start', 6), ('recolha', 7), ('recolha', 10), ('entrega', 7), ('entrega', 10), ('regresso', 6)]
Validação: Rota perfeitamente viável! Cumpre todas as restrições.

Greedy | heuristic_mst | Veículo 2
Rota: [('start', 6), ('recolha', 7), ('recolha', 10), ('entrega', 7), ('entrega', 10), ('regresso', 6)]
Validação: Rota perfeitamente viável! Cumpre todas as restrições.

A* | heuristic | Veículo 1
Rota: [('start', 6), ('recolha', 7), ('recolha', 10), ('entrega', 7), ('entrega', 10), ('regresso', 6)]
Validação: Rota perfeitamente viável! Cumpre todas as

,Algoritmo,Heurística,Veículo,Distância (km),Duração (min),Nós expandidos,Pesquisa (ms),Viável
0,Greedy,heuristic,1,171.71,311.3,6,0.755,True
1,Greedy,heuristic,2,171.71,311.3,6,0.423,True
2,Greedy,heuristic_mst,1,171.71,311.3,8,0.843,True
3,Greedy,heuristic_mst,2,171.71,311.3,8,0.824,True
4,A*,heuristic,1,171.71,311.3,9,0.495,True
5,A*,heuristic,2,171.71,311.3,9,0.629,True
6,A*,heuristic_mst,1,171.71,311.3,11,0.976,True
7,A*,heuristic_mst,2,171.71,311.3,11,1.246,True


#### 4.1.4 Análise crítica

**Qualidade da solução.** As oito combinações encontram a mesma rota: depósito 6, recolha do artigo 7, recolha do artigo 10, entrega do artigo 7, entrega do artigo 10 e regresso ao depósito 6. A distância é **171,71 km** e a duração é **311,30 minutos**, incluindo espera. Todas as soluções cumprem o validador. A* com ambas as heurísticas encontra a distância ótima neste cenário, pelas condições anteriores; a coincidência do resultado do Greedy não lhe confere uma garantia geral de optimalidade.

**Esforço de pesquisa.** Com a primeira heurística, Greedy expande **6 nós** e A* **9**; com a árvore mínima, expandem respetivamente **8** e **11**. Estes valores contam expansões de percursos, não localizações distintas. A primeira heurística orienta melhor a pesquisa nestas execuções. Greedy expande menos nós, mas isso não permite concluir que seja sempre melhor: pode encontrar rotas mais longas noutros cenários.

**Custo computacional.** Os tempos da tabela são observações de uma única execução. Dependem do ambiente, aquecimento e ordem das chamadas; não sustentam uma classificação de velocidade. A heurística de árvore mínima executa mais operações por avaliação, e a contagem de nós, por si só, não mede todo esse custo. Para estudar desempenho seriam necessárias medições repetidas e cenários maiores, fora do âmbito desta integração.

**Veículos e representatividade.** Os resultados coincidem porque ambos partem do mesmo depósito, usam a mesma matriz e têm capacidade muito superior aos 27,64 kg selecionados. A diferença de capacidade não condiciona este cenário. A experiência não demonstra que toda a procura possa ser servida, nem avalia refrigeração, coordenação de frota ou escalabilidade.

#### 4.1.5 Verificação do âmbito da Parte A

Foram integrados Greedy, A*, duas heurísticas admissíveis para o dataset atual, as oito combinações, rotas completas, custos, duração, expansões, tempos e validação de viabilidade. Os algoritmos são importados de `informed.py`; o modelo, os CSV e as demonstrações preliminares mantêm-se. O notebook deve ser executado desde o início, com as dependências do projeto instaladas e a raiz do repositório como diretório de trabalho.

Esta verificação cobre os requisitos relevantes transmitidos pelo grupo a partir das indicações do professor: Greedy e A*, duas heurísticas, rotas para os dois veículos, nós expandidos, comparação de soluções, custos e tempos, e análise crítica neste notebook. O enunciado original não foi consultado; não se afirma conformidade com exigências adicionais não comunicadas. As conclusões globais do projeto ficam reservadas à secção 5, após integração das restantes partes.

### 4.2 Pesquisa local — parte reservada

Hill-Climbing: a desenvolver pelo elemento responsável. Esta integração não altera nem antecipa essa implementação.

### 4.3 CSP e coordenação de frota — parte reservada

CSP Backtracking e coordenação de frota: a desenvolver pelo elemento responsável. Esta integração não altera nem antecipa essa implementação.

## 5. Result Analysis & Conclusion

*(A desenvolver na Entrega Final — Fase 2: Tabela comparativa de métricas, análise crítica, limitações e conclusões)*
